<a href="https://colab.research.google.com/github/JoseAlberto88/Hugging-Face-Text-Classification/blob/main/huggingface_text_classification_tutorial_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Hugging Face Text Classification Tutorial

* Resources notebook: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial
* Setup steps: https://www.learnhuggingface.com/extras/setup

**Note** A GPU is needed on Google Colab, go to Runtime -> Change runtime -> Hardware accelerator -> GPU.

### Import necessary libraries

In [2]:
import transformers

In [1]:
# Install dependencies (this is mostly for Google Colab)
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using torch version: {torch.__version__}" )
print(f"Using datasets version: {datasets.__version__}")


Using transformers version: 5.15.0
Using torch version: 2.11.0+cpu
Using datasets version: 5.0.1


## 3. Getting a dataset

Building food not food text classification model: need food not food text dataset.

In [3]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [4]:
# What features are there ?

dataset.column_names

{'train': ['text', 'label']}

In [5]:
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [6]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Import random samples


In [7]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f"Text: {text} | Label: {label}")

[INFO] Random samples from dataset:

Text: A close-up shot of a ripe and juicy peach with a sprinkle of cinnamon | Label: food
Text: Set of glasses gleaming in a cabinet | Label: not_food
Text: Decadent butter chicken curry, featuring tender chicken in a velvety sauce made with butter and tomatoes, served with warm naan bread. | Label: food
Text: Sushi with a spicy kick, featuring jalapeno peppers or spicy mayo. | Label: food
Text: A bowl of cherries with a sprig of mint for garnish | Label: food


In [8]:
# Get unique label values
dataset["train"].unique("label")

['food', 'not_food']

In [9]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])

Counter({'food': 125, 'not_food': 125})

In [10]:
# Turn our dataset into a DataFrame and get a random sample
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
124,"Celery in a bowl, served with a side of peanut...",food
120,A slice of pepperoni pizza with a layer of mel...,food
42,Set of tongs stored in a drawer,not_food
26,"Fresh cherry tomatoes in a basket, sprinkled w...",food
47,Wooden hanger holding clothes on a rack,not_food
238,Seasonal sushi roll with ingredients like pers...,food
78,Low-carb sushi roll with cucumber or seaweed w...,food


In [11]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

We want to:

1. Tokenize our text -> turn our text into numbers (this goes for labels as well).
2. Create a train/test split -> want to train our model on the training split and want to evaluate our model on the test split.

In [12]:
# Create a mapping for labels to numeric value

id2label = {0: "not_food", 1: "food"}
label2id = {"not_food" : 0, "food" : 1}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [13]:
# Create mappings programmatically from dataset
id2label = {idx : label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
id2label

{0: 'not_food', 1: 'food'}

In [14]:
label2id = {label : idx for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id

{'not_food': 0, 'food': 1}

In [15]:
# turn labels into 0 or 1

def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample = {"text" : "This is a sentence about my favprite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favprite food: honey', 'label': 1}

In [16]:
# Map our dataset labels to numbers (the whole thing)
# We do this with dataset.map()  - https://huggingface.co/docs/datasets/process#map
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [17]:
# Shuffle data and look at more 5 random examples
dataset.shuffle()[:5]

{'text': ['Sushi with a spicy kick, featuring jalapeno peppers or spicy mayo.',
  'Pizza with a white sauce base, topped with spinach and artichokes',
  'A pair of slices from a barbecue chicken pizza',
  'Set of forks kept in a holder',
  'Friends having a barbecue in the backyard while pets play nearbyA collection of cleaning products under the sink'],
 'label': [1, 1, 1, 0, 0]}

### Split the dataset into training and test sets

* Train set = model will learn patters on this dataset
* Validation set (optional) = we can tune our model's hyperparameters on this set
* Test set = model will evaluate patters on this dataset

We can split our dataset using `datasets.Dataset.train_test_split`. https://huggingface.co/docs/datasets/v4.8.4/process#split

In [18]:
# Split our dataset into train/test splits

dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [23]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

{'text': 'Crunchy sushi roll with a creamy filling, featuring shrimp tempura and avocado.',
 'label': 1}

In [26]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["test"][random_idx_test]
random_sample_test

{'text': 'Spicy chickpea curry bowl, featuring nutty chickpeas in a flavorful sauce with onions and tomatoes, finished with a wedge of lemon.',
 'label': 1}